# 10 - Prediction Pipeline

### Objective

Validate the StockVision prediction pipeline using the reusable `predict_stock()` function implemented in `src/prediction.py`.

The pipeline generates machine learning predictions for three forecasting horizons:

- **1D** → Next trading day
- **5D** → Next 5 trading days
- **20D** → Next 20 trading days

The prediction pipeline uses the latest available stock features and the corresponding horizon-specific trained model.

The pipeline is tested across all supported stocks and all three prediction horizons.

In [1]:
import sys
from pathlib import Path

# Put the PROJECT ROOT (not src/) on the path so `from src...` imports work
sys.path.append(str(Path.cwd().parent))

# Reload edited .py files automatically (no need to restart the kernel after editing src/)
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import FEATURES
from src.prediction import predict_stock

In [2]:
stocks = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "RELIANCE.NS",
    "TCS.NS",
    "HDFCBANK.NS",
    "INFY.NS"
]

## Model Features

The prediction pipeline uses the same feature definitions and feature order used during model training.

The feature list is imported from `src/config.py` to maintain consistency between training and prediction.

The same latest feature values are passed to the model for each horizon. However, each horizon uses its own separately trained model.

In [3]:
features = FEATURES

print("Number of features:", len(features))
print("Features:")
print(features)

Number of features: 18
Features:
['Adj Close', 'Adj_Daily_Return', 'Daily_Return', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'SMA_5', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'Volume_Change', 'High_Low_Range', 'Open_Close_Change', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist']


## Generate Predictions

Generate predictions for every supported stock across all three forecasting horizons.

For each stock:

- 1D model predicts next-day direction.
- 5D model predicts direction over the next 5 trading days.
- 20D model predicts direction over the next 20 trading days.

The reusable `predict_stock()` function loads the appropriate horizon-specific model automatically.

In [4]:
horizons = ["1D", "5D", "20D"]

all_predictions = []
    
for stock in stocks:

    for horizon in horizons:

        result = predict_stock(
            stock=stock,
            horizon=horizon
        )

        all_predictions.append(result)


predictions_df = pd.DataFrame(all_predictions)

predictions_df

,Stock,Horizon,Date,Prediction,Probability
0,AAPL,1D,2026-09-18,UP,51.13
1,AAPL,5D,2026-09-18,UP,53.92
2,AAPL,20D,2026-09-18,DOWN,57.61
3,MSFT,1D,2026-09-18,DOWN,51.43
4,MSFT,5D,2026-09-18,UP,56.61
5,MSFT,20D,2026-09-18,DOWN,78.88
6,NVDA,1D,2026-09-18,DOWN,52.56
7,NVDA,5D,2026-09-18,DOWN,58.41
8,NVDA,20D,2026-09-18,DOWN,61.31
9,AMZN,1D,2026-09-18,UP,53.83


In [5]:
print("Prediction Distribution by Horizon:")

print(
    pd.crosstab(predictions_df["Horizon"], predictions_df["Prediction"])
)

Prediction Distribution by Horizon:
Prediction  DOWN  UP
Horizon             
1D             2   6
20D            4   4
5D             3   5


In [6]:
print("Probability Summary by Horizon:")

print(
    predictions_df.groupby("Horizon")["Probability"].describe().round(2)
)

Probability Summary by Horizon:
         count   mean   std    min    25%    50%    75%    max
Horizon                                                       
1D         8.0  55.44  4.57  51.13  52.28  53.37  58.71  63.77
20D        8.0  63.34  8.54  53.29  56.81  63.02  67.78  78.88
5D         8.0  57.11  3.75  52.35  54.00  57.51  58.74  64.10


In [7]:
predictions_df[
    [
        "Stock",
        "Horizon",
        "Date",
        "Prediction",
        "Probability"
    ]
]

,Stock,Horizon,Date,Prediction,Probability
0,AAPL,1D,2026-09-18,UP,51.13
1,AAPL,5D,2026-09-18,UP,53.92
2,AAPL,20D,2026-09-18,DOWN,57.61
3,MSFT,1D,2026-09-18,DOWN,51.43
4,MSFT,5D,2026-09-18,UP,56.61
5,MSFT,20D,2026-09-18,DOWN,78.88
6,NVDA,1D,2026-09-18,DOWN,52.56
7,NVDA,5D,2026-09-18,DOWN,58.41
8,NVDA,20D,2026-09-18,DOWN,61.31
9,AMZN,1D,2026-09-18,UP,53.83


## Conclusion

The StockVision prediction pipeline was successfully validated using the reusable `predict_stock()` function implemented in `src/prediction.py`.

Predictions and prediction probabilities were generated for all eight supported stocks using the latest available market features and finalized stock-specific models.

The validated prediction module can now be directly used by the StockVision Streamlit dashboard.